# Notebook 09 — Media Consumption Layer (Pew NPORS / ATP)

## Objective

This notebook introduces the **media consumption layer** into the Market Kinetics structural population model.  
Building on the previous notebooks, the goal is to estimate how different segments of the U.S. population **consume information and media platforms**, and project those behaviors onto the synthetic structural population derived from the ACS.

At this stage of the pipeline, each individual in the modeled population already has:

- **Structural attributes** (ACS)
- **Psychological traits** inferred from GSS (Notebook 07)
- **Cluster membership** and cluster-level psychological signatures (Notebook 08)

This notebook adds the **information environment dimension**, capturing how different population segments interact with media and digital platforms.

---

## Data Sources

Media behavior variables are derived from **Pew Research Center surveys**, primarily:

- **National Public Opinion Reference Survey (NPORS)**
- **American Trends Panel (ATP)**

These surveys provide nationally representative measurements of:

- Social media platform usage  
- Frequency of platform engagement  
- News and information consumption behaviors  

The survey datasets contain **individual-level responses**, enabling modeling of media behavior as a function of demographic and socioeconomic characteristics.

---

## Methodological Approach

The workflow mirrors the methodology used in **Notebook 07 for psychological inference using GSS data**.

The process follows four main phases:

### Phase 1 — Media Feature Engineering

Relevant variables from the Pew dataset are identified and transformed into interpretable **media behavior traits**, such as:

- Platform usage (e.g., Facebook, YouTube, TikTok)
- Frequency of engagement
- Overall media engagement level
- Social media participation

Categorical features are standardized into modeling-ready variables.

---

### Phase 2 — Projection Logic from Pew to ACS

Using the Pew survey dataset, media behavior traits are linked to structural characteristics that can be aligned with ACS variables.

The objective is not to build a purely predictive model for its own sake, but to use Pew’s nationally representative survey data to estimate:

P(media_trait | structural profile)

where the structural profile is defined by variables available in both Pew and ACS, such as:

- Age group
- Education tier
- Income tier
- Race/ethnicity
- Employment status
- Household-related characteristics

This step creates a projection framework that maps observed media behaviors in Pew onto structurally similar individuals in the ACS-based synthetic population.

---

### Phase 3 — Projection to the Synthetic Population

The conditional media behavior distributions derived from Pew are projected onto the ACS structural population, following the same logic used previously for the GSS psychological layer.

As a result, each individual in the synthetic population receives probabilistic media behavior traits based on their structural similarity to respondents in the Pew survey.

This produces a population-scale media layer that can later be aggregated at the cluster level and interpreted alongside the psychological layer.

### Phase 4 — Cluster-Level Media Signatures

Media probabilities are aggregated at the **cluster level**, producing behavioral signatures that describe how each structural segment interacts with the information environment.

Cluster profiles will therefore include three complementary dimensions:

- **Structural characteristics** (ACS)
- **Psychological traits** (GSS inference)
- **Media behaviors** (Pew projection)

---

## Output

This notebook produces the media behavior enrichment of the structural population and cluster summaries, forming the final input layer for the generation of **behaviorally grounded audience segments and proto-personas**.

Expected output datasets:
09_mk_structural_media_population_v1.parquet
09_mk_cluster_media_signatures_v1.parquet


---

## Role in the Market Kinetics Architecture

With the completion of this step, the segmentation model integrates:

    - Structure (ACS)

    - Psychology (GSS)

    - Media Behavior (Pew)


This multi-layer representation allows each population cluster to be interpreted not only in demographic and attitudinal terms, but also in relation to **the information channels through which it can be reached and influenced**.

In [65]:
# Core libraries
import pandas as pd
import numpy as np

# Paths
from pathlib import Path

# Display settings
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [66]:
# Project root 
PROJECT_ROOT = Path("../")

DATA_RAW = PROJECT_ROOT / "data" / "societal_raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "societal_processed"

In [67]:
# Load structural + psychological population dataset

structural_path = PROJECT_ROOT / "data" / "societal_models" / "07_mk_structural_attitudes_population_v1.parquet"

df_population = pd.read_parquet(structural_path)

print("Population dataset loaded")
print("Shape:", df_population.shape)

df_population.head()

Population dataset loaded
Shape: (778466, 35)


,serialno,sporder,pwgtp,age_bin,sex_label,race_eth,edu_tier,emp_tier,income_tier_fixed,mar_tier,commute_tier,tenure,household_size,vehicle_count,puma,hhincome_tier,household_type,cluster,ideology__conservative,ideology__liberal,ideology__moderate,party_alignment__democrat,party_alignment__independent,party_alignment__other,party_alignment__republican,religiosity__high,religiosity__low,religiosity__medium,religiosity__none,life_satisfaction__not_happy,life_satisfaction__pretty_happy,life_satisfaction__very_happy,media_engagement__high,media_engagement__low,media_engagement__medium
0,2023HU1043211,2,58,35-44,Male,Black_NH,HS_or_less,Employed,0-19k,Never_Married,Car,No_Rent,2,0,4316,0-19k,housing_unit,6,0.0,0.444367,0.555633,1.0,0.0,0.0,0.0,0.555633,0.111092,0.166638,0.166638,0.0,0.444367,0.555633,0.133306,0.800042,0.066653
1,2019HU1076190,2,46,35-44,Male,Black_NH,HS_or_less,Employed,0-19k,Never_Married,Car,No_Rent,4,2,5922,20-49k,housing_unit,0,0.0,0.444367,0.555633,1.0,0.0,0.0,0.0,0.555633,0.111092,0.166638,0.166638,0.0,0.444367,0.555633,0.133306,0.800042,0.066653
2,2019GQ0046130,1,12,35-44,Male,Black_NH,HS_or_less,Other_Not_in_Labor_Force,0-19k,Never_Married,Missing,Group_Quarters,1,0,11300,group_quarters,group_quarters,6,0.0,0.444367,0.555633,1.0,0.0,0.0,0.0,0.555633,0.111092,0.166638,0.166638,0.0,0.444367,0.555633,0.133306,0.800042,0.066653
3,2019HU0403832,1,76,35-44,Male,Black_NH,HS_or_less,Employed,0-19k,Never_Married,Work_From_Home,Owner,5,2,2510,50-99k,housing_unit,0,0.0,0.444367,0.555633,1.0,0.0,0.0,0.0,0.555633,0.111092,0.166638,0.166638,0.0,0.444367,0.555633,0.133306,0.800042,0.066653
4,2019HU0277198,1,64,35-44,Male,Black_NH,HS_or_less,Employed,0-19k,Never_Married,Car,No_Rent,4,1,4607,20-49k,housing_unit,5,0.0,0.444367,0.555633,1.0,0.0,0.0,0.0,0.555633,0.111092,0.166638,0.166638,0.0,0.444367,0.555633,0.133306,0.800042,0.066653


In [68]:
# Load NPORS dataset
npors_path = PROJECT_ROOT / "data" / "societal_raw" / "pew_data" / "NPORS_2025_for_public_release_FINAL.csv"

df_npors = pd.read_csv(npors_path, low_memory=False)

print("NPORS dataset loaded")
print("Shape:", df_npors.shape)

df_npors.head()

NPORS dataset loaded
Shape: (5022, 65)


,RESPID,MODE,LANGUAGE,LANGUAGEINITIAL,STRATUM,INTERVIEW_START,INTERVIEW_END,ECON1MOD,ECON1BMOD,COMTYPE2,UNITY,CRIMESAFE,GOVPROTCT,MOREGUNIMPACT,FIN_SIT,VET1,VOL12_CPS,EMINUSE,INTMOB,INTFREQ,INTFREQ_COLLAPSED,HOME4NW2,BBHOME,SMUSE_FB,SMUSE_YT,SMUSE_X,SMUSE_IG,SMUSE_SC,SMUSE_WA,SMUSE_TT,SMUSE_RD,SMUSE_BSK,SMUSE_TH,SMUSE_TS,RADIO,DEVICE1A,SMART2,NHISLL,RELIG,RELIGCAT1,BORN,ATTENDPER,ATTENDONLINE2,RELIMP,PRAY,EDUCCAT,REGISTRATION,PARTY,PARTYLN,PARTYSUM,HISP,RACECMB,RACETHN,AGEGRP,AGECAT,BIRTHPLACE,GENDER,ADULTS,VOTED2024,VOTEGEN_POST,INC_SDT1,CREGION,METRO,BASEWT,WEIGHT
0,1470,2,1,NaN,10,2025-05-27,2025-05-27,4,2,3,2,2,2,1,3,4,2,1,1,2,2,1.0,2.0,2.0,1.0,2.0,2.0,2.0,2.0,2.0,1.0,2.0,2.0,2.0,1,1,1.0,2,12,3,2,6,6,4,7,1,1,3,2.0,2,2,4,4,11,4,1,2,1,1,2.0,2,4,1,0.802200,0.497038
1,2374,2,1,NaN,7,2025-05-01,2025-05-01,3,2,1,2,3,1,3,3,4,2,1,1,2,2,1.0,2.0,1.0,1.0,2.0,1.0,2.0,2.0,1.0,2.0,2.0,2.0,2.0,1,1,1.0,2,1,1,2,6,6,3,6,2,1,2,NaN,2,2,2,2,10,4,1,1,4,1,2.0,5,4,1,0.532494,0.307081
2,1177,3,1,10.0,5,2025-03-04,2025-03-04,2,1,3,2,2,1,2,3,4,2,1,1,2,2,1.0,2.0,1.0,1.0,1.0,1.0,2.0,1.0,2.0,2.0,2.0,2.0,1.0,1,1,1.0,2,1,1,1,5,1,1,2,2,1,1,NaN,1,2,1,1,10,4,1,2,2,1,1.0,2,4,1,1.249642,0.646850
3,15459,2,1,NaN,10,2025-05-05,2025-05-05,3,3,3,2,2,2,3,4,4,2,1,1,2,2,2.0,NaN,2.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,1,1,1.0,2,1,1,2,5,5,1,2,2,1,1,NaN,1,2,1,1,6,2,1,1,2,1,1.0,1,2,2,1.604401,1.311245
4,9849,1,1,9.0,9,2025-02-22,2025-02-22,2,1,2,2,3,1,2,1,4,1,1,1,2,2,1.0,2.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0,1,1,1.0,2,1,1,1,2,4,1,1,1,1,1,NaN,1,2,1,1,10,4,1,1,4,1,1.0,8,1,1,0.676351,0.241761


In [69]:
df_npors.columns

Index(['RESPID', 'MODE', 'LANGUAGE', 'LANGUAGEINITIAL', 'STRATUM', 'INTERVIEW_START', 'INTERVIEW_END', 'ECON1MOD', 'ECON1BMOD', 'COMTYPE2', 'UNITY', 'CRIMESAFE', 'GOVPROTCT', 'MOREGUNIMPACT',
       'FIN_SIT', 'VET1', 'VOL12_CPS', 'EMINUSE', 'INTMOB', 'INTFREQ', 'INTFREQ_COLLAPSED', 'HOME4NW2', 'BBHOME', 'SMUSE_FB', 'SMUSE_YT', 'SMUSE_X', 'SMUSE_IG', 'SMUSE_SC', 'SMUSE_WA', 'SMUSE_TT',
       'SMUSE_RD', 'SMUSE_BSK', 'SMUSE_TH', 'SMUSE_TS', 'RADIO', 'DEVICE1A', 'SMART2', 'NHISLL', 'RELIG', 'RELIGCAT1', 'BORN', 'ATTENDPER', 'ATTENDONLINE2', 'RELIMP', 'PRAY', 'EDUCCAT',
       'REGISTRATION', 'PARTY', 'PARTYLN', 'PARTYSUM', 'HISP', 'RACECMB', 'RACETHN', 'AGEGRP', 'AGECAT', 'BIRTHPLACE', 'GENDER', 'ADULTS', 'VOTED2024', 'VOTEGEN_POST', 'INC_SDT1', 'CREGION', 'METRO',
       'BASEWT', 'WEIGHT'],
      dtype='str')

In [70]:
df_npors.info()

<class 'pandas.DataFrame'>
RangeIndex: 5022 entries, 0 to 5021
Data columns (total 65 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   RESPID             5022 non-null   int64  
 1   MODE               5022 non-null   int64  
 2   LANGUAGE           5022 non-null   int64  
 3   LANGUAGEINITIAL    2768 non-null   float64
 4   STRATUM            5022 non-null   int64  
 5   INTERVIEW_START    5022 non-null   str    
 6   INTERVIEW_END      5022 non-null   str    
 7   ECON1MOD           5022 non-null   int64  
 8   ECON1BMOD          5022 non-null   int64  
 9   COMTYPE2           5022 non-null   int64  
 10  UNITY              5022 non-null   int64  
 11  CRIMESAFE          5022 non-null   int64  
 12  GOVPROTCT          5022 non-null   int64  
 13  MOREGUNIMPACT      5022 non-null   int64  
 14  FIN_SIT            5022 non-null   int64  
 15  VET1               5022 non-null   int64  
 16  VOL12_CPS          5022 non-null   

### Phase 1 — NPORS Variable Selection

The NPORS dataset contains many survey variables not relevant to the media projection task.

This step extracts the subset of variables required for:

• structural alignment with the ACS population model  
• measurement of media platform usage  
• survey weighting  

The resulting dataset forms the basis for feature engineering and probability estimation in subsequent phases.

In [71]:
npors_cols = [
    
    # structural variables
    "AGEGRP",
    "GENDER",
    "RACETHN",
    "EDUCCAT",
    "INC_SDT1",
    
    # internet access / connectivity
    "EMINUSE",
    "INTMOB",
    "INTFREQ",
    "INTFREQ_COLLAPSED",
    
    # platform usage
    "SMUSE_FB",
    "SMUSE_YT",
    "SMUSE_X",
    "SMUSE_IG",
    "SMUSE_SC",
    "SMUSE_WA",
    "SMUSE_TT",
    "SMUSE_RD",
    "SMUSE_BSK",
    "SMUSE_TH",
    "SMUSE_TS",
    
    # other media behavior
    "RADIO",
    
    # survey weight
    "WEIGHT"
]

npors = df_npors[npors_cols].copy()

print("NPORS modeling dataset created")
print("Shape:", npors.shape)
print("Columns:")
print(npors.columns.tolist())

npors.head()

NPORS modeling dataset created
Shape: (5022, 22)
Columns:
['AGEGRP', 'GENDER', 'RACETHN', 'EDUCCAT', 'INC_SDT1', 'EMINUSE', 'INTMOB', 'INTFREQ', 'INTFREQ_COLLAPSED', 'SMUSE_FB', 'SMUSE_YT', 'SMUSE_X', 'SMUSE_IG', 'SMUSE_SC', 'SMUSE_WA', 'SMUSE_TT', 'SMUSE_RD', 'SMUSE_BSK', 'SMUSE_TH', 'SMUSE_TS', 'RADIO', 'WEIGHT']


,AGEGRP,GENDER,RACETHN,EDUCCAT,INC_SDT1,EMINUSE,INTMOB,INTFREQ,INTFREQ_COLLAPSED,SMUSE_FB,SMUSE_YT,SMUSE_X,SMUSE_IG,SMUSE_SC,SMUSE_WA,SMUSE_TT,SMUSE_RD,SMUSE_BSK,SMUSE_TH,SMUSE_TS,RADIO,WEIGHT
0,11,2,4,1,2,1,1,2,2,2.0,1.0,2.0,2.0,2.0,2.0,2.0,1.0,2.0,2.0,2.0,1,0.497038
1,10,1,2,2,5,1,1,2,2,1.0,1.0,2.0,1.0,2.0,2.0,1.0,2.0,2.0,2.0,2.0,1,0.307081
2,10,2,1,2,2,1,1,2,2,1.0,1.0,1.0,1.0,2.0,1.0,2.0,2.0,2.0,2.0,1.0,1,0.646850
3,6,1,1,2,1,1,1,2,2,2.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,1,1.311245
4,10,1,1,1,8,1,1,2,2,1.0,1.0,1.0,1.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0,1,0.241761


In [72]:
# Convert column names to lowercase
npors.columns = npors.columns.str.lower()

print("Column names standardized to lowercase")
npors.columns

Column names standardized to lowercase


Index(['agegrp', 'gender', 'racethn', 'educcat', 'inc_sdt1', 'eminuse', 'intmob', 'intfreq', 'intfreq_collapsed', 'smuse_fb', 'smuse_yt', 'smuse_x', 'smuse_ig', 'smuse_sc', 'smuse_wa', 'smuse_tt',
       'smuse_rd', 'smuse_bsk', 'smuse_th', 'smuse_ts', 'radio', 'weight'],
      dtype='str')

In [73]:
# inspecting structural variables
structural_vars = [
    "agegrp",
    "gender",
    "racethn",
    "educcat",
    "inc_sdt1"
]

for col in structural_vars:
    print("\n", col)
    print(npors[col].value_counts(dropna=False))


 agegrp
agegrp
10    572
11    486
9     478
8     419
5     384
12    379
7     377
13    376
4     358
6     342
3     315
2     245
1     235
99     56
Name: count, dtype: int64

 gender
gender
2     2758
1     2194
3       45
99      25
Name: count, dtype: int64

 racethn
racethn
1     3304
3      757
2      512
5      211
4      159
99      79
Name: count, dtype: int64

 educcat
educcat
1     2215
2     1587
3     1175
99      45
Name: count, dtype: int64

 inc_sdt1
inc_sdt1
1     939
8     858
5     692
4     659
2     451
6     447
3     423
7     324
99    229
Name: count, dtype: int64


In [74]:
# agegrp -> age_bin (aligned to ACS bins)

age_map = {
    1: "18-24",

    2: "25-34",
    3: "25-34",

    4: "35-44",
    5: "35-44",

    6: "45-54",
    7: "45-54",

    8: "55-64",
    9: "55-64",

    10: "65+",
    11: "65+",
    12: "65+",
    13: "65+"
}

npors["age_bin"] = npors["agegrp"].map(age_map)

# validation
print(npors[["agegrp","age_bin"]].head())
print(npors["age_bin"].value_counts(dropna=False))

   agegrp age_bin
0      11     65+
1      10     65+
2      10     65+
3       6   45-54
4      10     65+
age_bin
65+      1813
55-64     897
35-44     742
45-54     719
25-34     560
18-24     235
NaN        56
Name: count, dtype: int64


In [75]:
# gender
gender_map = {
    1: "Male",
    2: "Female"
}

npors["sex_label"] = npors["gender"].map(gender_map)

# quick validation
print(npors[["gender","sex_label"]].head())
print(npors["sex_label"].value_counts(dropna=False))

   gender sex_label
0       2    Female
1       1      Male
2       2    Female
3       1      Male
4       1      Male
sex_label
Female    2758
Male      2194
NaN         70
Name: count, dtype: int64


In [76]:
df_population['race_eth'].value_counts()

race_eth
White_NH    475529
Hispanic    132510
Black_NH     91553
Asian_NH     46401
Other_NH     32473
Name: count, dtype: int64[pyarrow]

In [77]:
# RACETHN -> race_eth

race_map = {
    1: "White_NH",
    2: "Black_NH",
    3: "Hispanic",
    5: "Asian_NH",
    4: "Other_NH"
}

npors["race_eth"] = npors["racethn"].map(race_map)

# validation
print(npors[["racethn", "race_eth"]].head())
print(npors["race_eth"].value_counts(dropna=False))

   racethn  race_eth
0        4  Other_NH
1        2  Black_NH
2        1  White_NH
3        1  White_NH
4        1  White_NH
race_eth
White_NH    3304
Hispanic     757
Black_NH     512
Asian_NH     211
Other_NH     159
NaN           79
Name: count, dtype: int64


In [78]:
df_population["edu_tier"].value_counts()

edu_tier
HS_or_less      295657
Some_college    230905
Bachelor        156864
Graduate         95040
Name: count, dtype: int64[pyarrow]

In [79]:
df_npors['EDUCCAT'].value_counts()

EDUCCAT
1     2215
2     1587
3     1175
99      45
Name: count, dtype: int64

In [80]:
# INC_SDT1 -> income_tier_fixed

income_map = {
    1: "0-19k",      # <30k, imperfect but closest lower-income bin
    2: "20-49k",     # 30-40k
    3: "20-49k",     # 40-50k
    4: "50-99k",     # 50-70k
    5: "50-99k",     # 70-100k
    6: "100-199k",   # 100-125k
    7: "100-199k",   # 125-150k
    8: "200k+"       # 150k+, top-coded approximation
}

npors["income_tier_fixed"] = npors["inc_sdt1"].map(income_map)

# validation
print(npors[["inc_sdt1", "income_tier_fixed"]].head())
print(npors["income_tier_fixed"].value_counts(dropna=False))

   inc_sdt1 income_tier_fixed
0         2            20-49k
1         5            50-99k
2         2            20-49k
3         1             0-19k
4         8             200k+
income_tier_fixed
50-99k      1351
0-19k        939
20-49k       874
200k+        858
100-199k     771
NaN          229
Name: count, dtype: int64


Because NPORS reports education using a three-category scheme, the “College graduate+”
category was disaggregated into “Bachelor” and “Graduate” using donor proportions from the
adult structural population. The split was stratified by age, sex, race/ethnicity, and income
tier, with overall college-plus proportions used as fallback for sparse cells.

In [81]:
# Build donor proportions for Bachelor vs Graduate
# using the structural population dataset

donor_cols = ["age_bin", "sex_label", "race_eth", "income_tier_fixed", "edu_tier"]

df_donor = df_population[donor_cols].copy()
df_donor = df_donor[df_donor["edu_tier"].isin(["Bachelor", "Graduate"])].copy()

donor_split = (
    df_donor
    .groupby(["age_bin", "sex_label", "race_eth", "income_tier_fixed", "edu_tier"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# Ensure both columns exist
if "Bachelor" not in donor_split.columns:
    donor_split["Bachelor"] = 0
if "Graduate" not in donor_split.columns:
    donor_split["Graduate"] = 0

donor_split["college_total"] = donor_split["Bachelor"] + donor_split["Graduate"]
donor_split["p_bachelor"] = donor_split["Bachelor"] / donor_split["college_total"]
donor_split["p_graduate"] = donor_split["Graduate"] / donor_split["college_total"]

donor_split = donor_split[
    ["age_bin", "sex_label", "race_eth", "income_tier_fixed", "p_bachelor", "p_graduate"]
].copy()

print("Donor split table shape:", donor_split.shape)
donor_split.head()

Donor split table shape: (359, 6)


edu_tier,age_bin,sex_label,race_eth,income_tier_fixed,p_bachelor,p_graduate
0,18-24,Female,Asian_NH,0-19k,0.891129,0.108871
1,18-24,Female,Asian_NH,100-199k,0.826087,0.173913
2,18-24,Female,Asian_NH,20-49k,0.898305,0.101695
3,18-24,Female,Asian_NH,200k+,1.000000,0.000000
4,18-24,Female,Asian_NH,50-99k,0.858586,0.141414


In [82]:
# Initialize edu_tier for categories that map directly

edu_map_partial = {
    3: "HS_or_less",
    2: "Some_college"
}

npors["edu_tier"] = npors["educcat"].map(edu_map_partial)

print(npors["edu_tier"].value_counts(dropna=False))

edu_tier
NaN             2260
Some_college    1587
HS_or_less      1175
Name: count, dtype: int64


In [83]:
# Merge donor Bachelor/Graduate proportions onto NPORS

npors = npors.merge(
    donor_split,
    on=["age_bin", "sex_label", "race_eth", "income_tier_fixed"],
    how="left"
)

print(npors[["age_bin", "sex_label", "race_eth", "income_tier_fixed", "p_bachelor", "p_graduate"]].head())

  age_bin sex_label  race_eth income_tier_fixed  p_bachelor  p_graduate
0     65+    Female  Other_NH            20-49k    0.600000    0.400000
1     65+      Male  Black_NH            50-99k    0.543417    0.456583
2     65+    Female  White_NH            20-49k    0.597723    0.402277
3   45-54      Male  White_NH             0-19k    0.711016    0.288984
4     65+      Male  White_NH             200k+    0.367002    0.632998


In [84]:
# Overall fallback split among college-plus in the population

overall_counts = (
    df_donor["edu_tier"]
    .value_counts(normalize=True)
)

overall_p_bachelor = overall_counts.get("Bachelor", 0.0)
overall_p_graduate = overall_counts.get("Graduate", 0.0)

print("Fallback p_bachelor:", round(overall_p_bachelor, 4))
print("Fallback p_graduate:", round(overall_p_graduate, 4))

Fallback p_bachelor: 0.6227
Fallback p_graduate: 0.3773


In [85]:
# Probabilistic split for NPORS college graduate+ respondents
# using donor proportions and fixed seed for reproducibility

rng = np.random.default_rng(42)

mask_college_plus = npors["educcat"] == 1

# Fill missing donor probabilities with overall fallback
p_grad = npors.loc[mask_college_plus, "p_graduate"].fillna(overall_p_graduate)

# Random draw
u = rng.random(mask_college_plus.sum())

# Assign final 4-level education tier
npors.loc[mask_college_plus, "edu_tier"] = np.where(
    u < p_grad.to_numpy(),
    "Graduate",
    "Bachelor"
)

# validation
print(npors["edu_tier"].value_counts(dropna=False))
print(npors[npors["educcat"] == 1]["edu_tier"].value_counts(normalize=True, dropna=False))

edu_tier
Some_college    1587
Bachelor        1190
HS_or_less      1175
Graduate        1025
NaN               45
Name: count, dtype: int64
edu_tier
Bachelor    0.537246
Graduate    0.462754
Name: proportion, dtype: float64


In [86]:
college_plus = df_population[df_population["edu_tier"].isin(["Bachelor","Graduate"])]

college_plus["edu_tier"].value_counts(normalize=True).round(4)

edu_tier
Bachelor    0.6227
Graduate    0.3773
Name: proportion, dtype: double[pyarrow]

In [87]:
npors[npors["educcat"] == 1].groupby("age_bin")["edu_tier"].value_counts(normalize=True)

age_bin  edu_tier
18-24    Bachelor    0.866667
         Graduate    0.133333
25-34    Bachelor    0.662069
         Graduate    0.337931
35-44    Bachelor    0.504000
         Graduate    0.496000
45-54    Bachelor    0.522472
         Graduate    0.477528
55-64    Bachelor    0.547009
         Graduate    0.452991
65+      Graduate    0.519947
         Bachelor    0.480053
Name: proportion, dtype: float64

In [88]:
npors[npors["educcat"] == 1].groupby("income_tier_fixed")["edu_tier"].value_counts(normalize=True)

income_tier_fixed  edu_tier
0-19k              Bachelor    0.659420
                   Graduate    0.340580
100-199k           Bachelor    0.545833
                   Graduate    0.454167
20-49k             Bachelor    0.661157
                   Graduate    0.338843
200k+              Graduate    0.582840
                   Bachelor    0.417160
50-99k             Bachelor    0.579760
                   Graduate    0.420240
Name: proportion, dtype: float64

In [91]:
def binary_smuse(series):
    return series.map({1: 1, 2: 0}).astype("Int8")

In [92]:
# Internet access (EMINUSE)
npors["internet_access"] = (
    npors["eminuse"]
    .map({1: 1, 2: 0})
    .astype("Int8")
)

npors["internet_access"].value_counts(dropna=False)

internet_access
1       4760
0        242
<NA>      20
Name: count, dtype: Int64

In [93]:
# mobile internet access (INTMOB)
npors["mobile_internet_access"] = (
    npors["intmob"]
    .map({1: 1, 2: 0})
    .astype("Int8")
)

npors["mobile_internet_access"].value_counts(dropna=False)

mobile_internet_access
1       4700
0        297
<NA>      25
Name: count, dtype: Int64

In [96]:
# internet frequency (INTFREQ_COLLAPSED) --1 = Almost constantly, 2 = Several times a day, 3 = Daily / weekly, 4 = Rarely or never
npors["internet_frequency"] = (
    npors["intfreq_collapsed"]
    .replace({99: pd.NA})
    .astype("Int8")
)

npors["internet_frequency"].value_counts(dropna=False)

internet_frequency
2       2311
1       1856
3        522
4        286
<NA>      47
Name: count, dtype: Int64

In [97]:
# radio listener (RADIO) --> 1: YES, 0: NO
npors["radio_listener"] = (
    npors["radio"]
    .map({1: 1, 2: 0})
    .astype("Int8")
)

npors["radio_listener"].value_counts(dropna=False)


radio_listener
1       3728
0       1251
<NA>      43
Name: count, dtype: Int64

In [105]:
# Reusable function for Pew SMUSE_* platform variables
# 1 = Yes, use this
# 2 = No, don't use this
# 99 = Refused / Web blank

def map_smuse(series):
    return (
        series
        .map({1: 1, 2: 0})
        .astype("Int8")
    )

# Apply to all platform variables
platform_map = {
    "smuse_fb": "facebook",
    "smuse_yt": "youtube",
    "smuse_x": "x_twitter",
    "smuse_ig": "instagram",
    "smuse_sc": "snapchat",
    "smuse_wa": "whatsapp",
    "smuse_tt": "tiktok",
    "smuse_rd": "reddit",
    "smuse_bsk": "bluesky",
    "smuse_th": "threads",
    "smuse_ts": "truthsocial"
}

for raw_col, new_col in platform_map.items():
    npors[new_col] = map_smuse(npors[raw_col])

# quick check
npors[list(platform_map.values())].head()

,facebook,youtube,x_twitter,instagram,snapchat,whatsapp,tiktok,reddit,bluesky,threads,truthsocial
0,0,1,0,0,0,0,0,1,0,0,0
1,1,1,0,1,0,0,1,0,0,0,0
2,1,1,1,1,0,1,0,0,0,0,1
3,0,1,0,0,0,0,0,0,0,0,0
4,1,1,1,1,1,0,0,0,0,0,0


In [106]:
npors[list(platform_map.values())].mean().sort_values(ascending=False)

youtube        0.856251
facebook        0.72782
instagram      0.472141
whatsapp       0.321361
tiktok         0.306692
reddit         0.250211
snapchat        0.21056
x_twitter      0.191118
threads        0.078481
bluesky        0.050676
truthsocial    0.042773
dtype: Float64

In [108]:
# creati
media_features = [
    "internet_access",
    "mobile_internet_access",
    "internet_frequency",
    "radio_listener",
    "youtube",
    "facebook",
    "instagram",
    "tiktok",
    "whatsapp",
    "reddit",
    "snapchat",
    "x_twitter",
    "threads",
    "bluesky",
    "truthsocial"
]

In [109]:
structural_cols = [
    "age_bin",
    "sex_label",
    "race_eth",
    "edu_tier",
    "income_tier_fixed"
]

In [110]:
# helper 
def weighted_mean(series, weights):
    return (series * weights).sum() / weights.sum()

In [112]:
# Level 1 — full structural profile
rows = []

for keys, group in npors.groupby(structural_cols):
    weights = group["weight"]
    result = dict(zip(structural_cols, keys))

    for feat in media_features:
        result[feat] = weighted_mean(group[feat], weights)

    rows.append(result)

media_probs_full = pd.DataFrame(rows)

print("Full media probability table shape:", media_probs_full.shape)
media_probs_full.head()

Full media probability table shape: (751, 20)


,age_bin,sex_label,race_eth,edu_tier,income_tier_fixed,internet_access,mobile_internet_access,internet_frequency,radio_listener,youtube,facebook,instagram,tiktok,whatsapp,reddit,snapchat,x_twitter,threads,bluesky,truthsocial
0,18-24,Female,Asian_NH,Bachelor,0-19k,1.0,1.0,2.000000,0.0,1.0,1.0,1.0,0.000000,1.0,0.0,1.0,0.0,0.000000,0.0,0.0
1,18-24,Female,Asian_NH,Bachelor,20-49k,1.0,1.0,1.935107,0.0,1.0,1.0,1.0,0.064893,0.0,1.0,1.0,0.0,0.935107,0.0,0.0
2,18-24,Female,Asian_NH,Graduate,0-19k,1.0,1.0,1.000000,1.0,1.0,1.0,1.0,0.000000,1.0,0.0,1.0,0.0,0.000000,0.0,0.0
3,18-24,Female,Asian_NH,Graduate,20-49k,1.0,1.0,1.000000,0.0,1.0,1.0,1.0,1.000000,1.0,1.0,1.0,1.0,0.000000,0.0,0.0
4,18-24,Female,Asian_NH,HS_or_less,50-99k,1.0,1.0,1.000000,0.0,1.0,0.0,1.0,1.000000,1.0,1.0,1.0,0.0,1.000000,0.0,0.0


In [113]:
### Fallback strategies.
# Level 2 - no income
structural_cols_l2 = [
    "age_bin",
    "sex_label",
    "race_eth",
    "edu_tier"
]

rows = []

for keys, group in npors.groupby(structural_cols_l2):
    weights = group["weight"]
    result = dict(zip(structural_cols_l2, keys))

    for feat in media_features:
        result[feat] = weighted_mean(group[feat], weights)

    rows.append(result)

media_probs_l2 = pd.DataFrame(rows)

print("Level 2 table shape:", media_probs_l2.shape)
media_probs_l2.head()


#level 3 - no income, no education

structural_cols_l3 = [
    "age_bin",
    "sex_label",
    "race_eth"
]

rows = []

for keys, group in npors.groupby(structural_cols_l3):
    weights = group["weight"]
    result = dict(zip(structural_cols_l3, keys))

    for feat in media_features:
        result[feat] = weighted_mean(group[feat], weights)

    rows.append(result)

media_probs_l3 = pd.DataFrame(rows)

print("Level 3 table shape:", media_probs_l3.shape)
media_probs_l3.head()

# level 4 - age only
structural_cols_l4 = ["age_bin"]

rows = []

for keys, group in npors.groupby(structural_cols_l4):
    weights = group["weight"]

    if not isinstance(keys, tuple):
        keys = (keys,)

    result = dict(zip(structural_cols_l4, keys))

    for feat in media_features:
        result[feat] = weighted_mean(group[feat], weights)

    rows.append(result)

media_probs_l4 = pd.DataFrame(rows)

print("Level 4 table shape:", media_probs_l4.shape)
media_probs_l4.head()


# Level 5 - global baseline
global_result = {}

weights = npors["weight"]

for feat in media_features:
    global_result[feat] = weighted_mean(npors[feat], weights)

media_probs_global = pd.DataFrame([global_result])

print("Global table shape:", media_probs_global.shape)
media_probs_global.head()

Level 2 table shape: (226, 19)
Level 3 table shape: (60, 18)
Level 4 table shape: (6, 16)
Global table shape: (1, 15)


,internet_access,mobile_internet_access,internet_frequency,radio_listener,youtube,facebook,instagram,tiktok,whatsapp,reddit,snapchat,x_twitter,threads,bluesky,truthsocial
0,0.945728,0.936359,1.780349,0.703467,0.840767,0.707844,0.50021,0.367878,0.322839,0.262496,0.253081,0.208932,0.083556,0.039885,0.034678


In [114]:
# projecting back onto population dataset
df_media = df_population.copy()

df_media = df_media.merge(
    media_probs_full,
    on=structural_cols,
    how="left"
)

print("After full merge:", df_media[media_features].isna().mean().mean())

After full merge: 0.16802532159400665


In [115]:
# filling with level 2 probabilities
df_l2 = df_population[structural_cols_l2].merge(
    media_probs_l2,
    on=structural_cols_l2,
    how="left"
)

for feat in media_features:
    df_media[feat] = df_media[feat].fillna(df_l2[feat])

In [116]:
# filling with level 3 probabilities
df_l3 = df_population[structural_cols_l3].merge(
    media_probs_l3,
    on=structural_cols_l3,
    how="left"
)

for feat in media_features:
    df_media[feat] = df_media[feat].fillna(df_l3[feat])

In [117]:
# filling with level 4 probabilities
df_l4 = df_population[structural_cols_l4].merge(
    media_probs_l4,
    on=structural_cols_l4,
    how="left"
)

for feat in media_features:
    df_media[feat] = df_media[feat].fillna(df_l4[feat])

In [118]:
# filling with global baseline
for feat in media_features:
    df_media[feat] = df_media[feat].fillna(media_probs_global.loc[0, feat])

In [119]:
# validate coverage
df_media[media_features].isna().sum().sort_values(ascending=False)

internet_access           0
mobile_internet_access    0
internet_frequency        0
radio_listener            0
youtube                   0
facebook                  0
instagram                 0
tiktok                    0
whatsapp                  0
reddit                    0
snapchat                  0
x_twitter                 0
threads                   0
bluesky                   0
truthsocial               0
dtype: int64

In [120]:
# saving the population_level enriched dataset
output_path = PROJECT_ROOT / "data" / "societal_models" / "09_mk_structural_media_population_v1.parquet"

df_media.to_parquet(output_path, index=False)

print("Saved:", output_path)
print("Shape:", df_media.shape)

Saved: ../data/societal_models/09_mk_structural_media_population_v1.parquet
Shape: (778466, 50)


In [121]:
# cluster aggregation
cluster_media = (
    df_media
    .groupby("cluster")[media_features]
    .mean()
    .reset_index()
)

cluster_media.head()

,cluster,internet_access,mobile_internet_access,internet_frequency,radio_listener,youtube,facebook,instagram,tiktok,whatsapp,reddit,snapchat,x_twitter,threads,bluesky,truthsocial
0,0,0.975515,0.973922,1.621539,0.692464,0.903581,0.767819,0.555239,0.399591,0.389048,0.314454,0.293915,0.225434,0.092811,0.043748,0.036132
1,1,0.972201,0.961274,1.711239,0.720104,0.875223,0.747471,0.500474,0.351527,0.307640,0.288545,0.262749,0.209762,0.080529,0.045407,0.047209
2,2,0.943603,0.953908,1.759227,0.633435,0.875467,0.729205,0.560850,0.491912,0.418816,0.244214,0.329075,0.208813,0.103623,0.032683,0.033509
3,3,0.939365,0.939456,1.722651,0.635746,0.866500,0.721682,0.560767,0.451269,0.322277,0.285366,0.351015,0.207911,0.085113,0.041296,0.032475
4,4,0.880856,0.843643,2.252111,0.736497,0.677464,0.606904,0.239332,0.183918,0.198905,0.123932,0.095745,0.122655,0.042090,0.025587,0.037224


In [122]:
cluster_media.info()

<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   cluster                 7 non-null      uint16 
 1   internet_access         7 non-null      float64
 2   mobile_internet_access  7 non-null      float64
 3   internet_frequency      7 non-null      float64
 4   radio_listener          7 non-null      float64
 5   youtube                 7 non-null      float64
 6   facebook                7 non-null      float64
 7   instagram               7 non-null      float64
 8   tiktok                  7 non-null      float64
 9   whatsapp                7 non-null      float64
 10  reddit                  7 non-null      float64
 11  snapchat                7 non-null      float64
 12  x_twitter               7 non-null      float64
 13  threads                 7 non-null      float64
 14  bluesky                 7 non-null      float64
 15  trut

In [123]:
# saving cluster-level media signatures
cluster_output_path = PROJECT_ROOT / "data" / "societal_models" / "09_mk_cluster_media_signatures_v1.parquet"

cluster_media.to_parquet(cluster_output_path, index=False)

print("Saved:", cluster_output_path)
print("Shape:", cluster_media.shape)

Saved: ../data/societal_models/09_mk_cluster_media_signatures_v1.parquet
Shape: (7, 16)


## Notebook 09 — Summary

This notebook introduced the **media behavior layer** into the Market Kinetics structural population model using the Pew NPORS dataset.

Media platform usage and internet behavior variables were harmonized and engineered into interpretable indicators, including internet access, internet intensity, radio consumption, and usage of major digital platforms such as YouTube, Facebook, Instagram, TikTok, Reddit, and others.

Using the structurally aligned variables (age, sex, race/ethnicity, education, and income), weighted conditional probabilities of media behaviors were estimated from the NPORS survey data. These probabilities were then projected onto the synthetic ACS-based population, assigning media behavior likelihoods to each individual in the modeled population.

To ensure full population coverage, a hierarchical fallback strategy was implemented, progressively relaxing structural conditions when necessary.

The resulting outputs are:

- **09_mk_structural_media_population_v1.parquet**  
  Structural population enriched with media behavior probabilities.

- **09_mk_cluster_media_signatures_v1.parquet**  
  Cluster-level media behavior profiles summarizing the information environments associated with each structural segment.

With this step complete, the Market Kinetics population model now integrates three complementary layers:

- **Structural characteristics** (ACS)  
- **Psychological traits** (GSS inference)  
- **Media behavior patterns** (Pew NPORS)

These layers together enable the identification and interpretation of behaviorally grounded audience segments and their corresponding media ecosystems.